In [ ]:
"""
Async encoding with NATIVE backend (no ONNX).
Compare performance vs ONNX backend.
"""

from sentence_transformers import SentenceTransformer
from pathlib import Path
from time import time
import numpy as np
import os
import asyncio
from concurrent.futures import ThreadPoolExecutor
import torch

# 12-core optimization for native backend
os.environ['OMP_NUM_THREADS'] = "4"
os.environ['MKL_NUM_THREADS'] = "4"
os.environ['TORCH_NUM_THREADS'] = "4"

# PyTorch threading
torch.set_num_threads(4)
torch.set_num_interop_threads(1)



print("System Configuration (Native Backend - 12-core):")
print(f"  PyTorch threads: 12")
print(f"  PyTorch interop threads: 1")
print(f"  OMP/MKL threads: 12")
print(f"  Backend: Native (PyTorch)")
print(f"  Async: ENABLED\n")

RuntimeError: Error: cannot set number of interop threads after parallel work has started or set_num_interop_threads called

In [4]:
import pandas as pd
# df = pd.read_parquet("../ultrachat_200k_sft.parquet")

In [3]:
# df["prompt"].head()
# del df

In [ ]:

def get_sample_data(n=100000):
    """Generate sample sentences."""
    texts = [
        "The system processes user authentication requests securely.",
        "Machine learning models are trained on large datasets.",
        "Cloud computing enables scalable infrastructure.",
        "Data science explores patterns in datasets.",
        "Embeddings represent text as vectors.",
    ]
    df = pd.read_parquet("../ultrachat_200k_sft.parquet")  # Load data to ensure it's cached
    texts = df["prompt"].tolist()
    texts = texts[:n]
    del df
    # return (texts * (n // len(texts) + 1))[:n]
    return texts


class NativeAsyncEncoder:
    """Async encoder using native PyTorch backend."""
    
    def __init__(self, model_name='../model_files', num_workers=3):
        # Load with native backend (no ONNX)
        self.model = SentenceTransformer(model_name)
        # Ensure model is on CPU
        self.model = self.model.to('cpu')
        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        self.loop = asyncio.get_event_loop()
    
    async def encode_async(self, text_chunks, batch_size=256):
        """Encode text chunks asynchronously with native backend."""
        tasks = [
            self.loop.run_in_executor(
                self.executor,
                lambda chunk=chunk: self.model.encode(
                    chunk,
                    batch_size=batch_size,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    device='cpu'
                )
            )
            for chunk in text_chunks
        ]
        
        results = await asyncio.gather(*tasks)
        return np.vstack(results)


async def main():
    print("=" * 70)
    print("NATIVE BACKEND + ASYNC ENCODING (12-CORE OPTIMIZED)")
    print("=" * 70)
    
    # Load data
    texts = get_sample_data(100000)
    print(f"\nProcessing {len(texts)} sentences...\n")
    
    # Initialize encoder with native backend
    print("Loading model with NATIVE backend (PyTorch)...")
    encoder = NativeAsyncEncoder('../model_files', num_workers=3)
    print("✓ Model loaded\n")
    
    # Split into chunks for parallel processing
    chunk_size = len(texts) // 3
    text_chunks = [
        texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)
    ]
    
    print("Async encoding with NATIVE backend (4 chunks, 4 workers)...")
    start = time()
    
    embeddings = await encoder.encode_async(text_chunks, batch_size=256)
    
    elapsed = time() - start
    throughput = len(texts) / elapsed
    
    # Results
    print("\n" + "=" * 70)
    print("RESULTS (NATIVE BACKEND + ASYNC)")
    print("=" * 70)
    print(f"Total time: {elapsed:.2f}s")
    print(f"Throughput: {throughput:,.0f} sentences/sec")
    print(f"Output shape: {embeddings.shape}")
    print(f"Precision: {embeddings.dtype}")
    
    print(f"\nOptimizations active:")
    print(f"  ✓ 4 concurrent encoding tasks (ThreadPoolExecutor)")
    print(f"  ✓ Each task gets 3 cores (12÷4 = 3 cores/task)")
    print(f"  ✓ PyTorch num_threads=12")
    print(f"  ✓ PyTorch interop_threads=1")
    print(f"  ✓ Batch size 1024")
    print(f"  ✓ Device: CPU")
    print(f"  ✓ Backend: Native (not ONNX)")
    
    # Compare to ONNX
    onnx_throughput = 1303  # from previous run
    current_throughput = throughput
    comparison = current_throughput / onnx_throughput
    
    print(f"\n" + "=" * 70)
    print("NATIVE vs ONNX COMPARISON")
    print("=" * 70)
    print(f"ONNX throughput:    1,303 sent/s (baseline)")
    print(f"Native throughput:  {current_throughput:,.0f} sent/s")
    
    if comparison > 1:
        print(f"Native is {comparison:.2f}x FASTER than ONNX ✓")
    else:
        print(f"ONNX is {1/comparison:.2f}x faster than Native")
    
    return embeddings


if __name__ == "__main__":
    # Run async main
    # embeddings = asyncio.run(main())
    embeddings = await main()
    print(f"\n✓ Encoding complete!")


# # results from previous runs for comparison:
# System Configuration (Native Backend - 12-core):
#   PyTorch threads: 12
#   PyTorch interop threads: 1
#   OMP/MKL threads: 12
#   Backend: Native (PyTorch)
#   Async: ENABLED

# ======================================================================
# NATIVE BACKEND + ASYNC ENCODING (12-CORE OPTIMIZED)
# ======================================================================

# Processing 100000 sentences...

# Loading model with NATIVE backend (PyTorch)...
# ✓ Model loaded

# Async encoding with NATIVE backend (4 chunks, 4 workers)...

# ======================================================================
# RESULTS (NATIVE BACKEND + ASYNC)
# ======================================================================
# Total time: 77.64s
# Throughput: 1,288 sentences/sec
# Output shape: (100000, 384)
# Precision: float32

# Optimizations active:
#   ✓ 4 concurrent encoding tasks (ThreadPoolExecutor)
#   ✓ Each task gets 3 cores (12÷4 = 3 cores/task)
#   ✓ PyTorch num_threads=12
#   ✓ PyTorch interop_threads=1
#   ✓ Batch size 1024
#   ✓ Device: CPU
#   ✓ Backend: Native (not ONNX)

# ======================================================================
# NATIVE vs ONNX COMPARISON
# ======================================================================
# ONNX throughput:    1,303 sent/s (baseline)
# Native throughput:  1,288 sent/s
# ONNX is 1.01x faster than Native

# ✓ Encoding complete!

NATIVE BACKEND + ASYNC ENCODING (12-CORE OPTIMIZED)

Processing 100000 sentences...

Loading model with NATIVE backend (PyTorch)...
✓ Model loaded

Async encoding with NATIVE backend (4 chunks, 4 workers)...


In [ ]:
__name__

'__main__'

In [ ]:
"""
Async encoding with NATIVE backend (no ONNX).
Compare performance vs ONNX backend.
"""

from sentence_transformers import SentenceTransformer
from pathlib import Path
from time import time
import numpy as np
import os
import asyncio
from concurrent.futures import ThreadPoolExecutor
import torch
import pandas

# 12-core optimization for native backend
os.environ['OMP_NUM_THREADS'] = "4"
os.environ['MKL_NUM_THREADS'] = "4"
os.environ['TORCH_NUM_THREADS'] = "4"

# PyTorch threading
torch.set_num_threads(4)
torch.set_num_interop_threads(1)



print("System Configuration (Native Backend - 12-core):")
print(f"  PyTorch threads: 12")
print(f"  PyTorch interop threads: 1")
print(f"  OMP/MKL threads: 12")
print(f"  Backend: Native (PyTorch)")
print(f"  Async: ENABLED\n")


def get_sample_data(n=100000):
    """Generate sample sentences."""
    texts = [
        "The system processes user authentication requests securely.",
        "Machine learning models are trained on large datasets.",
        "Cloud computing enables scalable infrastructure.",
        "Data science explores patterns in datasets.",
        "Embeddings represent text as vectors.",
    ]
    df = pd.read_parquet("../ultrachat_200k_sft.parquet")  # Load data to ensure it's cached
    texts = df["prompt"].tolist()
    texts = texts[:n]
    del df
    # return (texts * (n // len(texts) + 1))[:n]
    return texts


class NativeAsyncEncoder:
    """Async encoder using native PyTorch backend."""
    
    def __init__(self, model_name='../model_files', num_workers=3):
        # Load with native backend (no ONNX)
        self.model = SentenceTransformer(model_name)
        # Ensure model is on CPU
        self.model = self.model.to('cpu')
        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        self.loop = asyncio.get_event_loop()
    
    async def encode_async(self, text_chunks, batch_size=256):
        """Encode text chunks asynchronously with native backend."""
        tasks = [
            self.loop.run_in_executor(
                self.executor,
                lambda chunk=chunk: self.model.encode(
                    chunk,
                    batch_size=batch_size,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    device='cpu'
                )
            )
            for chunk in text_chunks
        ]
        
        results = await asyncio.gather(*tasks)
        return np.vstack(results)


async def main():
    print("=" * 70)
    print("NATIVE BACKEND + ASYNC ENCODING (12-CORE OPTIMIZED)")
    print("=" * 70)
    
    # Load data
    texts = get_sample_data(100000)
    print(f"\nProcessing {len(texts)} sentences...\n")
    
    # Initialize encoder with native backend
    print("Loading model with NATIVE backend (PyTorch)...")
    encoder = NativeAsyncEncoder('../model_files', num_workers=3)
    print("✓ Model loaded\n")
    
    # Split into chunks for parallel processing
    chunk_size = len(texts) // 3
    text_chunks = [
        texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)
    ]
    
    print("Async encoding with NATIVE backend (4 chunks, 4 workers)...")
    start = time()
    
    embeddings = await encoder.encode_async(text_chunks, batch_size=256)
    
    elapsed = time() - start
    throughput = len(texts) / elapsed
    
    # Results
    print("\n" + "=" * 70)
    print("RESULTS (NATIVE BACKEND + ASYNC)")
    print("=" * 70)
    print(f"Total time: {elapsed:.2f}s")
    print(f"Throughput: {throughput:,.0f} sentences/sec")
    print(f"Output shape: {embeddings.shape}")
    print(f"Precision: {embeddings.dtype}")
    
    print(f"\nOptimizations active:")
    print(f"  ✓ 4 concurrent encoding tasks (ThreadPoolExecutor)")
    print(f"  ✓ Each task gets 3 cores (12÷4 = 3 cores/task)")
    print(f"  ✓ PyTorch num_threads=12")
    print(f"  ✓ PyTorch interop_threads=1")
    print(f"  ✓ Batch size 1024")
    print(f"  ✓ Device: CPU")
    print(f"  ✓ Backend: Native (not ONNX)")
    
    # Compare to ONNX
    onnx_throughput = 1303  # from previous run
    current_throughput = throughput
    comparison = current_throughput / onnx_throughput
    
    print(f"\n" + "=" * 70)
    print("NATIVE vs ONNX COMPARISON")
    print("=" * 70)
    print(f"ONNX throughput:    1,303 sent/s (baseline)")
    print(f"Native throughput:  {current_throughput:,.0f} sent/s")
    
    if comparison > 1:
        print(f"Native is {comparison:.2f}x FASTER than ONNX ✓")
    else:
        print(f"ONNX is {1/comparison:.2f}x faster than Native")
    
    return embeddings


if __name__ == "__main__":
    # Run async main
    # embeddings = asyncio.run(main())
    embeddings = await main()
    print(f"\n✓ Encoding complete!")

# My Tests

In [1]:
from sentence_transformers import SentenceTransformer
from time import time
import numpy as np
import os
import torch
import pandas as pd

# 12-core optimization for native backend
no_of_threads = 12  # Use all available cores for the single process
batch_size = 256

os.environ['OMP_NUM_THREADS'] = f"{no_of_threads}"
os.environ['MKL_NUM_THREADS'] = f"{no_of_threads}"
os.environ['TORCH_NUM_THREADS'] = f"{no_of_threads}"

torch.set_num_threads(no_of_threads)
torch.set_num_interop_threads(1)

d:\Dsoft\Projects\ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_sample_data(n=100000):
    """Generate sample sentences."""
    # Note: Ensure the path is correct for your local machine
    df = pd.read_parquet(r"D:\Dsoft\Projects\ML\notebooks\ultrachat_200k_sft.parquet")
    texts = df["prompt"].tolist()[:n]
    return texts

In [5]:
# 1. Load data
print("Loading Data Sample")
texts = get_sample_data(100000)
texts.sort(key = lambda x: len(x))
print(f"\nLoaded {len(texts)} sentences.")

# 2. Initialize encoder
print("Loading model...")
model = SentenceTransformer('../model_files', device='cpu')
print("✓ Model loaded\n")

# 3. Sequential Encoding
print(f"Encoding {len(texts)} sentences sequentially (Batch Size: {batch_size})...")

start = time()
print("Starting model eval")
model.eval()
print(f"Eval done in {time() - start}")

print("Starting inference for embedding creations")
start = time()
with torch.inference_mode():
    # In sequential mode, we pass the whole list to the model
    # SentenceTransformer handles internal batching automatically
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

elapsed = time() - start
throughput = len(texts) / elapsed

# 4. Results
print("\n" + "=" * 70)
print("RESULTS (SEQUENTIAL)")
print("=" * 70)
print(f"batch_size: {batch_size}")
print(f"Total time: {elapsed:.2f}s")
print(f"Throughput: {throughput:,.0f} sentences/sec")
print(f"Output shape: {embeddings.shape}")


Loading Data Sample

Loaded 100000 sentences.
Loading model...
✓ Model loaded

Encoding 100000 sentences sequentially (Batch Size: 256)...
Starting model eval
Eval done in 0.0004291534423828125
Starting inference for embedding creations


Batches: 100%|██████████| 391/391 [20:33<00:00,  3.16s/it]



RESULTS (SEQUENTIAL)
batch_size: 256
Total time: 1237.69s
Throughput: 81 sentences/sec
Output shape: (100000, 384)


In [29]:
len(texts)

100000

In [30]:
with torch.inference_mode():
    # In sequential mode, we pass the whole list to the model
    # SentenceTransformer handles internal batching automatically
    embeddings = model.encode(
        texts,
        batch_size=128,
        show_progress_bar=True,
        # convert_to_numpy=True
    )

Batches:   6%|▋         | 50/782 [03:52<56:37,  4.64s/it]  


KeyboardInterrupt: 

In [7]:
embeddings

array([[-0.01921735,  0.05312051,  0.01903816, ..., -0.00240842,
        -0.04326284,  0.07299219],
       [ 0.13278054, -0.04673428,  0.00833236, ..., -0.01558725,
        -0.03494466, -0.02067245],
       [ 0.05926463,  0.08412746,  0.05030655, ...,  0.03195616,
        -0.01850193,  0.05573095],
       ...,
       [-0.08500507,  0.03460276, -0.00634193, ..., -0.03960776,
         0.00154365,  0.04047323],
       [ 0.00527049,  0.02349216,  0.0206551 , ..., -0.07500561,
         0.08150277,  0.02166477],
       [-0.03930785,  0.11843572,  0.05067811, ...,  0.0497327 ,
        -0.03273105,  0.00579942]], shape=(10000, 384), dtype=float32)

In [8]:
texts

["These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?",
 'Which famous landmarks should I visit in London, beyond the usual ones?',
 "Write a comprehensive blog post of at least 1000 words about the top 10 most eco-friendly cities in the world and their renewable energy initiatives. Use a formal and informative tone, and include statistics, case studies, and expert opinions to support your claims. Make sure to cover various aspects of sustainability, such as publi

In [25]:
test_sentence = ["What are the rules and restrictions in place for COVID-19 in the city?"]
torch.set_float32_matmul_precision('highest')
em_1 = model.encode(test_sentence)
torch.set_float32_matmul_precision('high')
em_2 = model.encode(test_sentence)


In [21]:
sa = np.concatenate([em_1, em_2], axis=0)

In [22]:
sa

array([[ 6.30809963e-02,  6.25148565e-02, -9.37203411e-03,
        -5.00127450e-02,  3.29169184e-02,  5.44526279e-02,
        -6.04108796e-02, -8.65189824e-03, -1.08311556e-01,
         2.67361645e-02,  1.00800931e-01, -2.20366735e-02,
        -2.44986955e-02,  7.14506283e-02, -1.46110179e-02,
        -1.32995620e-02,  3.59194539e-02, -1.03900477e-01,
        -4.53393068e-03, -1.84145365e-02,  6.12252019e-02,
        -1.99399665e-02, -1.11014675e-02, -2.55959593e-02,
        -4.33718003e-02,  3.81290317e-02, -3.42154689e-02,
         5.70454486e-02,  4.14639786e-02,  5.61269373e-02,
         3.40484753e-02, -1.32210627e-02,  1.05789462e-02,
         1.75171776e-03, -1.26383891e-02, -4.75470908e-02,
         1.00973971e-01, -4.51138653e-02,  4.67203110e-02,
         5.84355444e-02,  3.86836119e-02, -2.16685105e-02,
         4.67127785e-02,  2.43231803e-02,  6.29022121e-02,
         3.16705257e-02, -1.06504828e-01,  5.79654202e-02,
         5.02988137e-03, -6.94949403e-02,  4.36213836e-0

In [26]:
em_1

array([[ 6.30809963e-02,  6.25148565e-02, -9.37203411e-03,
        -5.00127450e-02,  3.29169184e-02,  5.44526279e-02,
        -6.04108796e-02, -8.65189824e-03, -1.08311556e-01,
         2.67361645e-02,  1.00800931e-01, -2.20366735e-02,
        -2.44986955e-02,  7.14506283e-02, -1.46110179e-02,
        -1.32995620e-02,  3.59194539e-02, -1.03900477e-01,
        -4.53393068e-03, -1.84145365e-02,  6.12252019e-02,
        -1.99399665e-02, -1.11014675e-02, -2.55959593e-02,
        -4.33718003e-02,  3.81290317e-02, -3.42154689e-02,
         5.70454486e-02,  4.14639786e-02,  5.61269373e-02,
         3.40484753e-02, -1.32210627e-02,  1.05789462e-02,
         1.75171776e-03, -1.26383891e-02, -4.75470908e-02,
         1.00973971e-01, -4.51138653e-02,  4.67203110e-02,
         5.84355444e-02,  3.86836119e-02, -2.16685105e-02,
         4.67127785e-02,  2.43231803e-02,  6.29022121e-02,
         3.16705257e-02, -1.06504828e-01,  5.79654202e-02,
         5.02988137e-03, -6.94949403e-02,  4.36213836e-0

In [27]:
em_2

array([[ 6.30809963e-02,  6.25148565e-02, -9.37203411e-03,
        -5.00127450e-02,  3.29169184e-02,  5.44526279e-02,
        -6.04108796e-02, -8.65189824e-03, -1.08311556e-01,
         2.67361645e-02,  1.00800931e-01, -2.20366735e-02,
        -2.44986955e-02,  7.14506283e-02, -1.46110179e-02,
        -1.32995620e-02,  3.59194539e-02, -1.03900477e-01,
        -4.53393068e-03, -1.84145365e-02,  6.12252019e-02,
        -1.99399665e-02, -1.11014675e-02, -2.55959593e-02,
        -4.33718003e-02,  3.81290317e-02, -3.42154689e-02,
         5.70454486e-02,  4.14639786e-02,  5.61269373e-02,
         3.40484753e-02, -1.32210627e-02,  1.05789462e-02,
         1.75171776e-03, -1.26383891e-02, -4.75470908e-02,
         1.00973971e-01, -4.51138653e-02,  4.67203110e-02,
         5.84355444e-02,  3.86836119e-02, -2.16685105e-02,
         4.67127785e-02,  2.43231803e-02,  6.29022121e-02,
         3.16705257e-02, -1.06504828e-01,  5.79654202e-02,
         5.02988137e-03, -6.94949403e-02,  4.36213836e-0

In [4]:
import os
os.listdir()

['model_comparison.py',
 'onnx_advanced.py',
 'onnx_async_pretok.py',
 'onnx_fast.py',
 'onnx_native_async.py',
 'onnx_optimizations.md',
 'onnx_optimization_results.ipynb',
 'onnx_test.py',
 'onnx_ultra_optimized.py',
 'onnx_vs_native_comparison.ipynb',
 'testnote.ipynb',
 'test_pytorch.py',
 'test_pytorch2.py']